# NCBI Virus Dataset Creation

Author: Alexander Maksiaev

Purpose: Create dataset using NCBI, by de-duplicating from Andersen, and relabeling sequences. 

In [1]:
# Housekeeping

import os
import pandas as pd
import dateutil
import re
import importlib
import utils  
importlib.reload(utils)
from utils import * 

In [34]:
# Paths

# Dates
start_date = "11-01-2021"
end_date = "07-18-2025"
date_range = start_date + "--" + end_date

# home = "C:/Users/maksi/Documents/Statistics/Projects/Avian_Flu_Files/"
home = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/"
downloads = home + "NCBI_Virus/downloads/" + date_range + "_Antarctica_North_America_South_America/"
temp_files = home + "NCBI_Virus/temp/"
complete_files = home + "NCBI_Virus/complete/" # + date_range + "_Antarctica_North_America_South_America/" 

# andersen_dedup = home + "Andersen/complete/11-01-2021--06-13-2025/" # Use the previous date range for de-duplication!
# andersen_dedup2 = home + "Andersen/complete/06-14-2025--07-04-2025/"
andersen = home + "Andersen/complete/" # 11-01-2021--07-04-2025/"
# andersen_dedup = andersen
combined_files = home + "Combinations/Andersen_NCBI_Virus/" + date_range + "_Antarctica_North_America_South_America/" 

# references = "C:/Users/maksi/Documents/Statistics/projects/Avian_Flu/references/"
references = "C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu/references/"
os.chdir(references)
states_ref = pd.read_csv("states_ref.csv")

# genotypes_df = pd.read_excel("genotype_key.xlsx")
# genotypes = list(genotypes_df["Genotype"])

genotypes = "D1.1" # ["B3.13", "D1.1", "D1.3"]

os.chdir(downloads)

## De-Duplication

In [17]:
metadata = pd.read_csv("sequences.csv")
print(len(metadata))

# Make sure we only have completed sequences -- 8 segments each 

metadata_counts = metadata.groupby(metadata.SRA_Accession, as_index=False).size()
# print(metadata_counts)
metadata_counted = metadata.merge(metadata_counts, on="SRA_Accession")

# Only keep those with size >= 8

metadata_complete_segs = metadata_counted[metadata_counted["size"] >= 8] # May have duplicates
metadata_complete_segs = metadata_complete_segs.drop_duplicates(subset="GenBank_Title", keep="first") # Get rid of duplicate segments

# Now only accept == 8 segments

metadata_segments = metadata_complete_segs[metadata_complete_segs["size"] == 8]
# metadata_segments

120949


In [18]:
# De-duplicate from Andersen using SRA Accession

# If even one SRA Accession in this list exists in the NCBI Virus dataframe, remove it from NCBI Virus dataframe
andersen_sras = []
# Grab files
for dirpath, dirs, files in os.walk(andersen):
    for file in files:
        file_name = os.path.join(dirpath, file)
        if ".fasta" in file_name:
            fasta_file = fasta_df_complete(file_name, states_ref) # Convert fasta file to dataframe
            sra_accessions = fasta_file["Identifier"]
            for value in sra_accessions.values:
                if "SRR" in value:
                    andersen_sras.append(value)
            isolates = fasta_file["Isolate_Id"]
            for value in isolates.values:
                andersen_sras.append(value)
    break 

# Grab more files
# for dirpath, dirs, files in os.walk(andersen_dedup):
#     for file in files:
#         file_name = os.path.join(dirpath, file)
#         if ".fasta" in file_name:
#             fasta_file = fasta_df_complete(file_name, states_ref) # Convert fasta file to dataframe
#             sra_accessions = fasta_file["Identifier"]
#             for value in sra_accessions.values:
#                 if "SRR" in value:
#                     andersen_sras.append(value)
            
# Remove duplicates from Andersen
for value in andersen_sras: # to remove
    if "SRR" in value:
        metadata_segments = metadata_segments[metadata_segments["SRA_Accession"] != value]
    else:
        metadata_segments = metadata_segments[metadata_segments["Isolate"] != value]
    # print(value)
    
print(len(metadata_segments))
# metadata_segments
# print(count)

28568


## Add sequences to dataframe

In [19]:
# NCBI Virus Naming Convention Example:
# Influenza A virus |USA: IN|25-006338-001-original|H5N1|2025-02-20|Meleagris gallopavo|GenBank|SRR33124721|SAMN47941411|PRJNA980729|Influenza A virus (A/Turkey/IN/25-006338-001-original/2025(H5N1)) segment 1 polymerase PB2 (PB2) gene, complete cds
# Organism_Name | Geo_Location | Isolate | Genotype | Collection_Date | Host | GenBank_RefSeq | SRA_Accession | BioSample | BioProject | GenBank_Title

# Get sequences and headers together
headers = []
isolates = []
sras = []
headers_seqs = {}

sequences_fasta = df_from_fasta("sequences.fasta") # Turn fasta into a dataframe

# Filter dataframe to only include filtered SRA accessions
sequences_fasta = sequences_fasta[sequences_fasta["full_header"].str.contains("|".join(list(metadata_segments["SRA_Accession"].values)))]
sequences_fasta["SRA_Accession"] = sequences_fasta["full_header"].apply(lambda x: x.split("|")[-4])
# Extract segment number so that we can add the correct sequences to the correct sample
sequences_fasta["Segment"] = sequences_fasta["full_header"].apply(lambda x: int(re.search(r'segment (.?) ', x.split("|")[-1]).group(1)))

# Double-check the de-duplication
print(len(sequences_fasta)) 
print(len(metadata_segments))

# Add sequences to the dataframe
metadata_segments = pd.merge(metadata_segments, sequences_fasta, on=["SRA_Accession", "Segment"])

28568
28568


In [6]:
metadata_segments

,Accession,GenBank_RefSeq,Assembly,SRA_Accession,BioSample,BioProject,Organism_Name,Species,Genus,Family,...,Submitters,Organization,Org_location,Publications,Collection_Date,Release_Date,Molecule_type,size,full_header,sequence
0,PV949028.1,GenBank,NaN,SRR34270180,SAMN49684856,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Aufderhar,M., Franzen,K., Killian,M.L., Lantz,...",USDA Animal Plant Health Inspection Service-Na...,USA,NaN,2025-05-12,2025-07-18,ssRNA(-),8,>Influenza A virus |USA: ID|25-015748-001-orig...,GAATAAAAGAACTGAGAGATCTAATGTCACAGTCTCGCACTCGCGA...
1,PV949029.1,GenBank,NaN,SRR34270180,SAMN49684856,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Aufderhar,M., Franzen,K., Killian,M.L., Lantz,...",USDA Animal Plant Health Inspection Service-Na...,USA,NaN,2025-05-12,2025-07-18,ssRNA(-),8,>Influenza A virus |USA: ID|25-015748-001-orig...,CTCTTCTTGAAAGTTCCAGCGCAAAATGCCATAAGCACCACATTCC...
2,PV949030.1,GenBank,NaN,SRR34270180,SAMN49684856,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Aufderhar,M., Franzen,K., Killian,M.L., Lantz,...",USDA Animal Plant Health Inspection Service-Na...,USA,NaN,2025-05-12,2025-07-18,ssRNA(-),8,>Influenza A virus |USA: ID|25-015748-001-orig...,CTTTGTGCGACAATGCTTCAATCCAATGATTGTCGAGCTTGCGGAA...
3,PV949031.1,GenBank,NaN,SRR34270180,SAMN49684856,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Aufderhar,M., Franzen,K., Killian,M.L., Lantz,...",USDA Animal Plant Health Inspection Service-Na...,USA,NaN,2025-05-12,2025-07-18,ssRNA(-),8,>Influenza A virus |USA: ID|25-015748-001-orig...,ATGGAGAACATAGTACTACTTCTTGCAATAGTTAGCCTTGTTAAAA...
4,PV949032.1,GenBank,NaN,SRR34270180,SAMN49684856,PRJNA1102327,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Aufderhar,M., Franzen,K., Killian,M.L., Lantz,...",USDA Animal Plant Health Inspection Service-Na...,USA,NaN,2025-05-12,2025-07-18,ssRNA(-),8,>Influenza A virus |USA: ID|25-015748-001-orig...,ATGGCGTCTCAAGGCACCAAACGATCCTATGAACAAATGGAAACTG...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28563,OQ565628.1,GenBank,GCA_039342815.1,SRR23852495,SAMN33745292,PRJNA944237,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Fernandez-Diaz,M., Villanueva-Perez,D., Tataje...","FARVET S.A.C., Laboratorio de Bioinformatica",Peru,1.0,2022-12,2023-03-08,ssRNA(-),8,>Influenza A virus |Peru|VFAR-140|H5N1|2022-12...,GTCAAAATGGAGAACATAGTACTACTTCTTGCAATAGTTAGCCTTG...
28564,OQ565629.1,GenBank,GCA_039342815.1,SRR23852495,SAMN33745292,PRJNA944237,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Fernandez-Diaz,M., Villanueva-Perez,D., Tataje...","FARVET S.A.C., Laboratorio de Bioinformatica",Peru,1.0,2022-12,2023-03-08,ssRNA(-),8,>Influenza A virus |Peru|VFAR-140|H5N1|2022-12...,TAGATAATCACTCACTGAGTGRYATSCACATCATGGCRTMYCARGR...
28565,OQ565630.1,GenBank,GCA_039342815.1,SRR23852495,SAMN33745292,PRJNA944237,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Fernandez-Diaz,M., Villanueva-Perez,D., Tataje...","FARVET S.A.C., Laboratorio de Bioinformatica",Peru,1.0,2022-12,2023-03-08,ssRNA(-),8,>Influenza A virus |Peru|VFAR-140|H5N1|2022-12...,CCATTGGATCARTCTGTATGGTAATTGGGATAGTCAGYTTGATGCT...
28566,OQ565631.1,GenBank,GCA_039342815.1,SRR23852495,SAMN33745292,PRJNA944237,Influenza A virus,Alphainfluenzavirus influenzae,Alphainfluenzavirus,Orthomyxoviridae,...,"Fernandez-Diaz,M., Villanueva-Perez,D., Tataje...","FARVET S.A.C., Laboratorio de Bioinformatica",Peru,1.0,2022-12,2023-03-08,ssRNA(-),8,>Influenza A virus |Peru|VFAR-140|H5N1|2022-12...,TTGAAAGATGAGTCTTCTAACCGAGGTCGAAACGTACGTTCTCTCT...


## Create FASTA files using deduplicated sequences

In [20]:
# Create 1 fasta file per header
metadata_segments["Partial_Header"] = metadata_segments["full_header"].apply(lambda x: "|".join(x.split("|")[:-1]))

# Create list of dataframes
df_list = []
for partial_header in list(set(metadata_segments["Partial_Header"].values)): # Unique partial headers only
    # Get smaller dataframe
    df = metadata_segments[metadata_segments["Partial_Header"] == partial_header]
    df = df.sort_values(by="Segment")
    # Make sure there are 8 segments
    if len(df) == 8:
        # Forbidden characters in headers
        for c in ["/", "|", " ", ":", ",", "(", ")", "-"]:
            df["full_header"] = df["full_header"].apply(lambda x: x.replace(c, "_"))
            df_list.append(df)

# Make fasta files
for df in df_list:
    # Forbidden characters in file name 
    title = df["Partial_Header"].values[0]
    for c in [">", "/", "|", " ", ":", ",", "(", ")", "-"]:
        title = title.replace(c, "_")
    df_to_fasta(df, title + ".fasta", temp_files)

## Re-Labeling Using GenoFlu

**STOP HERE AND USE GENOFLU TO FIND GENOTYPES.** Then, make sure "output.tsv" is in the downloads directory.

In [21]:
# Merging

os.chdir(downloads)

# Read in genoflu results
output_genoflu1 = pd.read_csv("output1.tsv", delimiter="\t")
output_genoflu2 = pd.read_csv("output2.tsv", delimiter="\t")
output_genoflu = pd.concat([output_genoflu1, output_genoflu2]).drop_duplicates(keep="first")

# Re-name partial headers to be exact the same as the file name, including extension 
partial_headers = []
for partial_header in metadata_segments["Partial_Header"].values:
    for c in [">", "/", "|", " ", ":", ",", "(", ")", "-"]:
        partial_header = partial_header.replace(c, "_")
    partial_headers.append(partial_header + ".fasta")
metadata_segments["File Name"] = partial_headers

metadata_genoflu = metadata_segments.merge(output_genoflu, how="inner", on="File Name")

# Fill the rest of the 8 segments with the same genotype
metadata_genoflu = metadata_genoflu.ffill()

print(metadata_genoflu)


       Accession GenBank_RefSeq         Assembly SRA_Accession     BioSample  \
0     PV949028.1        GenBank              NaN   SRR34270180  SAMN49684856   
1     PV949029.1        GenBank              NaN   SRR34270180  SAMN49684856   
2     PV949030.1        GenBank              NaN   SRR34270180  SAMN49684856   
3     PV949031.1        GenBank              NaN   SRR34270180  SAMN49684856   
4     PV949032.1        GenBank              NaN   SRR34270180  SAMN49684856   
...          ...            ...              ...           ...           ...   
1451  PV792597.1        GenBank  GCA_050967725.1   SRR33476663  SAMN48398151   
1452  PV792598.1        GenBank  GCA_050967725.1   SRR33476663  SAMN48398151   
1453  PV792599.1        GenBank  GCA_050967725.1   SRR33476663  SAMN48398151   
1454  PV792600.1        GenBank  GCA_050967725.1   SRR33476663  SAMN48398151   
1455  PV792601.1        GenBank  GCA_050967725.1   SRR33476663  SAMN48398151   

        BioProject      Organism_Name  

C:\Users\maksiaevai.NCBI_NT\AppData\Local\Temp\1\ipykernel_29812\3808481145.py:21: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  metadata_genoflu = metadata_genoflu.ffill()


We want:
* Host
* Geo-Location
* Isolate
* Year
* Collection Date
* Host Type
* Genotype

In [22]:
# Animals 

os.chdir(references)
animals_ref = pd.read_csv("animals_ref.csv")

# Create animals ref if needed

unique_animals_all = sort_animals_andersen(metadata_genoflu)

# Flatten unique_animals_all
every_unique_animal = []
for animal in unique_animals_all:
    every_unique_animal.append(animal)

print(every_unique_animal)

unique_animals_set = list(set(every_unique_animal)) # Get rid of duplicates

# If animal not in ref1, put in ref2

common_animals = []
# Check if animals in unique_animals_set are in ref1
for animal in unique_animals_set:
    for col in animals_ref.columns:
        if animal in animals_ref[col].values and type(animal) == str:
            common_animals.append(animal)

# If not in ref1, make a list of the new animals
different_animals = []
for animal in unique_animals_set:
    if animal not in common_animals:
        different_animals.append(animal)

print(different_animals)

# Add to dataframe
animals_df = animals_ref
# Make different_animals same length as dataframe, if shorter
if len(different_animals) < len(animals_df):
    number_of_times_to_add_nan = len(animals_df) - len(different_animals)
    for i in range(number_of_times_to_add_nan):
        different_animals.append(float('nan'))
# If longer, deal with that later

animals_df["new"] = (different_animals)

print(animals_df)

animals_df.to_csv("animals_ref_to_sort.csv") # Make sure name is different to avoid overwriting the first reference 


['bos taurus', 'meleagris gallopavo', 'anatidae', 'gallus gallus']
[]
           wild_avian domestic_avian               cattle        feline  \
0    great_horned_owl       pheasant            dairy_cow           cat   
1        common_raven         turkey               cattle  domestic_cat   
2       cooper's_hawk        chicken  cattle milk product     feral_cat   
3        coopers_hawk          goose          bovine_milk        feline   
4             peafowl    guinea_fowl              bovine   domestic-cat   
..                ...            ...                  ...           ...   
790     laughing gull            NaN                  NaN           NaN   
791      caspian tern            NaN                  NaN           NaN   
792       eared grebe            NaN                  NaN           NaN   
793          flamingo            NaN                  NaN           NaN   
794    swanson's_hawk            NaN                  NaN           NaN   

      other_mammal       huma

In [23]:
# Re-Labeling

os.chdir(references)
animals_ref = pd.read_csv("animals_ref.csv")

# Make sure NaN doesn't mess up the whole name
metadata_genoflu = metadata_genoflu.fillna("")
metadata_genoflu["Host"] = metadata_genoflu["Host"].apply(str.lower)

# Fix animals
fix_animals_andersen(metadata_genoflu, animals_ref)

# Get the years
metadata_genoflu["Years"] = metadata_genoflu["Collection_Date"].apply(lambda x: dateutil.parser.parse(x).strftime("%Y"))

# Make new labels
names = ">" + metadata_genoflu["SRA_Accession"] + "|A/" + metadata_genoflu["Host"].apply(lambda x: x.replace(" ", "_")) + "/" + metadata_genoflu["Geo_Location"].apply(lambda x: x.split(": ")[-1]) + "/" + metadata_genoflu["Isolate"] + "/" + metadata_genoflu["Years"].apply(lambda x: str(x)) + "|" + metadata_genoflu["Genotype_x"] + "|" + metadata_genoflu["Geo_Location"].apply(lambda x: x.replace(": ", "-")) + "|" + metadata_genoflu["Collection_Date"] + "|" + metadata_genoflu["Host_Type"] + "|" + metadata_genoflu["Genotype_y"]

metadata_genoflu["Name"] = names

print(metadata_genoflu["Name"])


0       >SRR34270180|A/bos_taurus/ID/25-015748-001-ori...
1       >SRR34270180|A/bos_taurus/ID/25-015748-001-ori...
2       >SRR34270180|A/bos_taurus/ID/25-015748-001-ori...
3       >SRR34270180|A/bos_taurus/ID/25-015748-001-ori...
4       >SRR34270180|A/bos_taurus/ID/25-015748-001-ori...
                              ...                        
1451    >SRR33476663|A/gallus_gallus/ND/25-013300-001-...
1452    >SRR33476663|A/gallus_gallus/ND/25-013300-001-...
1453    >SRR33476663|A/gallus_gallus/ND/25-013300-001-...
1454    >SRR33476663|A/gallus_gallus/ND/25-013300-001-...
1455    >SRR33476663|A/gallus_gallus/ND/25-013300-001-...
Name: Name, Length: 1456, dtype: object


## Rename segments and make complete FASTA files

In [28]:
# Set up segments

segments = {1:"PB2", 2:"PB1", 3:"PA", 4:"HA", 5:"NP", 6:"NA", 7:"MP", 8:"NS"}
metadata_genoflu["Segment_Name"] = metadata_genoflu["Segment"].map(segments)

# Separate into several dataframes based on genotype + segment
segment_genotype_dfs = []
for segment in segments.values():
    m_g = metadata_genoflu[metadata_genoflu["Segment_Name"] == segment]
    for genotype in list(set(m_g["Genotype_y"].values)):
        if genotype in genotypes:
            df = m_g[(m_g["Genotype_y"] == genotype)] 
            segment_genotype_dfs.append(df)
            pair = genotype + "_" + segment
            print(pair)

D1.1_PB2
D1.1_PB1
D1.1_PA
D1.1_HA
D1.1_NP
D1.1_NA
D1.1_MP
D1.1_NS


In [31]:
print(metadata_genoflu[metadata_genoflu["Genotype_y"] == "D1.1"])

       Accession GenBank_RefSeq         Assembly SRA_Accession     BioSample  \
1440  PV792578.1        GenBank  GCA_050967665.1   SRR33476668  SAMN48398146   
1441  PV792579.1        GenBank  GCA_050967665.1   SRR33476668  SAMN48398146   
1442  PV792580.1        GenBank  GCA_050967665.1   SRR33476668  SAMN48398146   
1443  PV792581.1        GenBank  GCA_050967665.1   SRR33476668  SAMN48398146   
1444  PV792582.1        GenBank  GCA_050967665.1   SRR33476668  SAMN48398146   
1445  PV792583.1        GenBank  GCA_050967665.1   SRR33476668  SAMN48398146   
1446  PV792584.1        GenBank  GCA_050967665.1   SRR33476668  SAMN48398146   
1447  PV792585.1        GenBank  GCA_050967665.1   SRR33476668  SAMN48398146   
1448  PV792594.1        GenBank  GCA_050967725.1   SRR33476663  SAMN48398151   
1449  PV792595.1        GenBank  GCA_050967725.1   SRR33476663  SAMN48398151   
1450  PV792596.1        GenBank  GCA_050967725.1   SRR33476663  SAMN48398151   
1451  PV792597.1        GenBank  GCA_050

In [25]:
# Create FASTA files

os.chdir(complete_files)

names = []

for df in segment_genotype_dfs:
    print(df)
    if len(df["Genotype_y"].values[0]) > 0:
        # print(df)
        file_name = df["Genotype_y"].values[0] + "_" + df["Segment_Name"].values[0] + "_" + date_range + ".fasta"
        output_file = open(complete_files + file_name, "w")

        for index, row in df.iterrows():
            name = df.loc[index, "Name"]
            name = name.replace(" ", "_")
            names.append(name)
            sequence = df.loc[index, "sequence"]
            # First is header, second is sequence
            output_file.write(name + "\n")
            output_file.write(sequence + "\n")
        output_file.close()

       Accession GenBank_RefSeq         Assembly SRA_Accession     BioSample  \
1440  PV792578.1        GenBank  GCA_050967665.1   SRR33476668  SAMN48398146   
1448  PV792594.1        GenBank  GCA_050967725.1   SRR33476663  SAMN48398151   

       BioProject      Organism_Name                         Species  \
1440  PRJNA980729  Influenza A virus  Alphainfluenzavirus influenzae   
1448  PRJNA980729  Influenza A virus  Alphainfluenzavirus influenzae   

                    Genus            Family  ... Genotype_y  \
1440  Alphainfluenzavirus  Orthomyxoviridae  ...       D1.1   
1448  Alphainfluenzavirus  Orthomyxoviridae  ...       D1.1   

                            Genotype List Used, >=98.0%  \
1440  PB2:am24, PB1:ea3, PA:am4, HA:ea3, NP:am13, NA...   
1448  PB2:am24, PB1:ea3, PA:am4, HA:ea3, NP:am13, NA...   

                             Genotype Sample Title List  \
1440  am24:24-030039-001:PB2, ea3:22-013001-001:PB1,...   
1448  am24:24-030039-001:PB2, ea3:22-013001-001:PB1,... 

In [35]:
# Concatenate with new Andersen sequences

os.chdir(combined_files)

# andersen = home + "Andersen/complete/" + date_range + "/"

# Andersen files
filenames_andersen = []
for genotype in genotypes:
    # print(gisaid_andersen + genotype.replace(".", "_") + "/")
    # for dirpath, dirs, files in os.walk(gisaid_andersen + genotype.replace(".", "_") + "/" + date_range + "_" + genotype.replace(".", "_") + "/"):
    for dirpath, dirs, files in os.walk(andersen): # Find the fasta file
        for file in files:
            file_name = os.path.join(dirpath, file) # Get file name
            filenames_andersen.append(file_name)
        break 

# NCBI Virus files
filenames_ncbi = []
for dirpath, dirs, files in os.walk(complete_files): # Find the fasta file
    for file in files:
        file_name = os.path.join(dirpath, file) # Get file name
        filenames_ncbi.append(file_name)
    break 

print(filenames_ncbi)

common_genotypes = set()
# Concatenate the two -- should not have any overlap due to dates and deduplication 
for a_file in filenames_andersen:
    partial_filename_a = a_file.split("_")[-4].split("/")[-1] + "_" + a_file.split("_")[-3]
    for nv_file in filenames_ncbi:
        partial_filename_nv = nv_file.split("_")[-3].split("/")[-1] + "_" + nv_file.split("_")[-2]
        if partial_filename_a == partial_filename_nv:
            common_genotypes.add(partial_filename_a)
            filenames = [a_file, nv_file]
            with open(combined_files + partial_filename_a + "_combined_" + date_range + ".fasta", 'w') as outfile:
                for fname in filenames:
                    with open(fname) as infile:
                        for line in infile:
                            outfile.write(line)
                        infile.close()
                outfile.close()

print(common_genotypes)

for a_file in filenames_andersen:
    partial_filename_a = a_file.split("_")[-4].split("/")[-1] + "_" + a_file.split("_")[-3]
    # If genotype not found in NCBI Virus, include it as well
    if partial_filename_a not in common_genotypes:
        with open(combined_files + partial_filename_a + "_combined_" + date_range + ".fasta", 'w') as outfile2:
            with open(a_file) as infile2:
                for line in infile2:
                    outfile2.write(line)
                infile2.close()
            outfile2.close()

['C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/NCBI_Virus/complete/D1.1_HA_11-01-2021--07-18-2025.fasta', 'C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/NCBI_Virus/complete/D1.1_MP_11-01-2021--07-18-2025.fasta', 'C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/NCBI_Virus/complete/D1.1_NA_11-01-2021--07-18-2025.fasta', 'C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/NCBI_Virus/complete/D1.1_NP_11-01-2021--07-18-2025.fasta', 'C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/NCBI_Virus/complete/D1.1_NS_11-01-2021--07-18-2025.fasta', 'C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/NCBI_Virus/complete/D1.1_PA_11-01-2021--07-18-2025.fasta', 'C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/NCBI_Virus/complete/D1.1_PB1_11-01-2021--07-18-2025.fasta', 'C:/Users/maksiaevai.NCBI_NT/Documents/Avian_Flu_Files/NCBI_Virus/complete/D1.1_PB2_11-01-2021--07-18-2025.fasta']
{'D1.1_PA', 'D1.1_MP', 'D1.1_PB2', 'D1.1_NS', 'D1.1_NP', 'D1.1_NA', 'D1.1_HA', 'D1.1_